# 🎙️ F5-TTS Turkish — Zero-Shot Voice Cloning in 10 Seconds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beydah/voice-clone_f5-tts_tr/blob/main/colab/Voice_Cloning_Turkish_F5_TTS_EN.ipynb)
[![Website](https://img.shields.io/badge/Website-beydahsaglam.com-blue?style=flat&logo=googlechrome)](https://beydahsaglam.com)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-beydah-0A66C2?style=flat&logo=linkedin)](https://linkedin.com/in/beydah)
[![GitHub](https://img.shields.io/badge/GitHub-beydah-181717?style=flat&logo=github)](https://github.com/beydah)

This Google Colab workspace uses the **F5-TTS (Flow Matching)** architecture with community-trained Turkish weights (`marduk-ra/F5-TTS-Turkish`) to clone any voice with zero-shot accuracy from a **single 10-second reference audio sample**.

### 🚀 Why F5-TTS & Flow Matching?
- **Zero-Shot (No Training):** No need for hours of recordings, days of GPU rental, or LoRA/fine-tuning.
- **Natural & Fluent:** Captures vocal timbre, cadence, and breath naturally without robotic artifacts or diffusion whispers.
- **5 Simple Steps:** Prepare reference sample, upload, denoise, enter text, and download your studio clone!

In [ ]:
#@title ⚙️ Step 0: Install Dependencies (Click to Run)
#@markdown Installs F5-TTS, Vocos vocoder, and audio processing tools (~40 seconds).

import subprocess

print("⏳ Installing packages (f5-tts, pydub, soundfile, scipy)...")
subprocess.run(["pip", "install", "-q", "f5-tts", "pydub", "soundfile", "scipy"], check=True)

# Colab torchvision/transformers pipeline patch
import transformers
transformers.pipeline = lambda *args, **kwargs: None

print("✅ Setup completed! Proceed to Step 1.")

---
## 🎯 Step 1: Prepare Your Reference Audio

The model clones your voice timbre and cadence from an **8-12 second** reference recording.

### 🎙️ 29-Letter Turkish Golden Pangram:
> **"Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir."**

*(This sentence contains all 29 letters of the Turkish alphabet, enabling the model to learn your full phonetic vocal range in one pass.)*

### 💡 Audio Recording Tips:
1. **Duration:** Aim for 8 to 12 seconds.
2. **Microphone Placement:** Position the mic 15-20 cm away, slightly off-axis to avoid plosive breath pops.
3. **Tone:** Speak naturally, with clear enunciation and confident energy as if explaining an idea to a colleague.

In [ ]:
#@title 📤 Step 2: Upload Reference Audio
#@markdown Run this cell and select your recorded audio file (.wav, .mp3, .ogg, .m4a):

import os
from google.colab import files

print("🎙️ Please select your 8-12 second reference audio file:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("⚠️ Please upload an audio file!")

raw_audio_path = list(uploaded.keys())[0]
print(f"✅ Audio uploaded: {raw_audio_path}")

In [ ]:
#@title 🎛️ Step 3: Automated Audio Cleaning & Normalization
#@markdown Trims leading/trailing silence, applies a 75 Hz high-pass rumble filter, and normalizes to -1.0 dBFS.

import soundfile as sf
import numpy as np
from scipy import signal
from pydub import AudioSegment
from IPython.display import Audio, display

# Convert any input format to 48kHz mono WAV
seg = AudioSegment.from_file(raw_audio_path)
seg = seg.set_channels(1).set_frame_rate(48000)
temp_wav = "temp_raw.wav"
seg.export(temp_wav, format="wav")

data, sr = sf.read(temp_wav)
if len(data.shape) > 1:
    data = data.mean(axis=1)

# 1. 75 Hz High-pass Butterworth filter
sos = signal.butter(4, 75, 'hp', fs=sr, output='sos')
filtered = signal.sosfilt(sos, data)

# 2. Voice Activity Detection (VAD) & Trimming
frame_len = int(sr * 0.03)  # 30ms windows
energy = np.array([np.sum(filtered[i:i+frame_len]**2) for i in range(0, len(filtered)-frame_len, frame_len)])
thresh = np.percentile(energy, 65) * 0.08
active_frames = np.where(energy > thresh)[0]

if len(active_frames) > 0:
    start_idx = max(0, active_frames[0] * frame_len - int(sr * 0.08))
    end_idx = min(len(filtered), (active_frames[-1] + 1) * frame_len + int(sr * 0.12))
    trimmed = filtered[start_idx:end_idx]
else:
    trimmed = filtered

# 3. Soft edge fades (prevents clicks)
fade_in = int(sr * 0.015)
fade_out = int(sr * 0.025)
if len(trimmed) > (fade_in + fade_out):
    trimmed[:fade_in] *= np.linspace(0, 1, fade_in)
    trimmed[-fade_out:] *= np.linspace(1, 0, fade_out)

# 4. Normalization (-1.0 dBFS peak)
peak = np.max(np.abs(trimmed))
if peak > 0:
    trimmed = trimmed / peak * 0.89

clean_ref_path = "clean_reference.wav"
sf.write(clean_ref_path, trimmed, sr)
print(f"✨ Audio cleaned successfully! Duration: {len(trimmed)/sr:.2f} seconds")
print("🎧 Listen to clean reference audio:")
display(Audio(clean_ref_path, rate=sr))

---
## 🧠 Step 4: Enter Target Text & Clone Voice

Paste the text you want to synthesize in the **target_text** field below.

* **Zero-Hallucination Sentence Chunking:** The script splits long text into sentences, synthesizes each, and adds **250ms natural breath pauses** between sentences.
* **Phonetic Corrections:** Technical loanwords (React, HTML, CSS, API) are pre-mapped to smooth Turkish phonetics (`Riyekt`, `He-Te-Me-Le`).

In [ ]:
#@title 🚀 Step 4: Synthesize Voice with F5-TTS
#@markdown Set inference parameters:

reference_text = "Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir." #@param {type:"string"}
speech_speed = 1.0 #@param {type:"slider", min:0.8, max:1.3, step:0.02}
cfg_fidelity = 2.2 #@param {type:"slider", min:1.5, max:3.0, step:0.1}

#@markdown ---
#@markdown 📝 **Target Text to Synthesize:**
target_text = """
Bir modelin cevabını ekranda göstermek ile modelin arayüz kodunu çalıştırmasına izin vermek farklı kararlardır. Kişisel bir uygulamada ilk deneme için küçük, önceden tanımlanmış bileşenler ve doğrulanmış veri daha denetlenebilir bir başlangıç sunar.
"""

import os
import re
import time
import torch
import soundfile as sf
import numpy as np
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

# If text.txt was uploaded, read from it
if os.path.exists("text.txt"):
    with open("text.txt", "r", encoding="utf-8") as f:
        target_text = f.read()

# 1. Load Model (cached in memory)
if 'f5tts' not in globals():
    print("⏳ Loading F5-TTS Turkish model weights into GPU memory...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="f5_tts_turkish_1000000.safetensors")
    vocab_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="vocab.txt")
    
    f5tts = F5TTS(
        model='F5TTS_Base',
        ckpt_file=ckpt_path,
        vocab_file=vocab_path,
        device=device
    )
    print(f"✅ Model loaded! Compute device: {device.upper()}")

# 2. Phonetic Replacements
phonetics = {
    "React": "Riyekt",
    "HTML": "He-Te-Me-Le",
    "JavaScript": "Cavasikript",
    "DOM": "Dom",
    "CSS": "Si-Es-Es",
    "Next.js": "Nekst ceyes",
    "API": "Eypiyay",
    "JSON": "Ceysın",
    "Suspense": "Saspens",
    "fallback": "folbek",
    "transition": "trenzışın",
    "effect": "efekt",
    "render": "rendır",
    "URL": "Yu-Ar-El"
}

processed_text = target_text
for term, rep in phonetics.items():
    processed_text = processed_text.replace(term, rep)

ref_audio = clean_ref_path if 'clean_ref_path' in globals() else 'clean_reference.wav'

# 3. Sentence Splitting & Inference
sentences = [c.strip() for c in re.split(r'(?<=[.!?])\s+', processed_text.replace('\n', ' ')) if len(c.strip()) > 3]
print(f"🎙️ Synthesis started ({len(sentences)} sentences)...")

audio_chunks = []
sr_val = 24000
t0 = time.time()

for i, s in enumerate(sentences, 1):
    print(f"[{i}/{len(sentences)}] Generating: \"{s[:45]}...\"")
    wav, sr, _ = f5tts.infer(
        ref_file=ref_audio,
        ref_text=reference_text,
        gen_text=s,
        nfe_step=32,
        cfg_strength=cfg_fidelity,
        speed=speech_speed,
        seed=42,
        remove_silence=False
    )
    sr_val = sr
    audio_chunks.append(wav)
    # 250ms natural breath pause
    audio_chunks.append(np.zeros(int(0.25 * sr)))

final_wav = np.concatenate(audio_chunks)
output_file = "cloned_voice_output.wav"
sf.write(output_file, final_wav, sr_val)

duration = len(final_wav) / sr_val
elapsed = time.time() - t0
print(f"\n🎉 Synthesis complete! Duration: {duration:.1f} s | Wall time: {elapsed:.1f} s")

In [ ]:
#@title 📥 Step 5: Play & Download Audio
#@markdown Listen to the generated voice and download the output audio:

from IPython.display import Audio, display
from google.colab import files

display(Audio(output_file, rate=sr_val))
files.download(output_file)
print("✅ Download started!")

---
### 🔗 Links & Resources

- 🌐 **Author Website & Portfolio:** [beydahsaglam.com](https://beydahsaglam.com)
- 💼 **LinkedIn:** [linkedin.com/in/beydah](https://linkedin.com/in/beydah)
- 🐙 **GitHub:** [github.com/beydah](https://github.com/beydah)
- 📄 **F5-TTS Paper:** [F5-TTS: A Fairytaler that Fakes Fluent and Faithful Speech with Flow Matching](https://arxiv.org/abs/2410.06885)
- 🤗 **Model Weights:** [marduk-ra/F5-TTS-Turkish](https://huggingface.co/marduk-ra/F5-TTS-Turkish)